# WS4 — The Myopic Prescription: A Bayesian Contextual Bandit Under an Honest OPE Gate

**Workstream 4 of the pitch-sequencing rigor ladder — the first prescriptive rung (Phase B).**
This notebook builds one uncertainty-aware **"best next pitch" target policy** — a feasibility-masked
**Thompson policy** over WS3's counterfactual value grid `q̂(s, a)` — and evaluates it **offline**
through the off-policy-evaluation (OPE) gate. It never learns online and refits nothing: it
**consumes** WS3's saved artifacts (decision D33) and scores strictly through `eval/ope`.

## From prediction to prescription — and why the gate comes first

WS1–WS3 (Phase A) asked whether ordered history *predicts* selection and outcomes (findings #1/#2).
WS4 opens **Phase B** and takes the first step toward finding #3 — *what should he throw, and can we
prove it?* The discipline SPEC §0.3 imposes on that step is the reason this notebook is built the way
it is (verbatim):

> **OPE before policy.** Build and self-test the off-policy-evaluation harness (it must recover the
> observed policy's value) *before* optimizing any policy. Otherwise you produce recommendations you
> cannot evaluate.

So the first thing below the setup is the **gate**: behavior-policy recovery, run *first*. If the OPE
machinery cannot even recover the *observed* policy's value, the pipeline prints `FAILED_GATE` and
**stops before any target value is reported** — nothing downstream is interpretable.

## The chapter's actual headline: the myopic ceiling

WS4's result is an **honest INCONCLUSIVE with real scientific content**. On the positive synthetic
world (a planted velocity-transition whiff boost) the sequencing-prescription gap is directionally
positive but its confidence interval straddles zero — and that is *correct*, not a failure. The
planted effect is a **state-value** effect: its trigger `|velo_{t−1} − velo_{t−2}| ≥ 5` is fixed by
the history, so on a triggered pitch **every** current family inherits the same `+0.032` boost. A
**myopic** bandit can exploit only the small *family-differential* (`~0.012` spread), worth a
best-case `~0.003` run — **below the OPE noise floor** (`~0.006` CI half-width at synthetic scale).
The rest is the **setup** — throwing the pitch that *creates* the transition a pitch early — which
pays off in a *future* state a one-step value cannot see. WS4 *measures* this ceiling, and a
gap-machinery self-test proves the near-zero gap is the effect's property, not a blind detector. That
turns WS4's INCONCLUSIVE into the falsifiable target (decision D40) the sequential rungs — WS5 (a
tabular MDP that values a setup pitch) and WS7 (offline RL) — exist to **exceed**.

## The three findings this notebook keeps separate (SPEC §0, verbatim)

> 1. **Selection structure** — prior pitches help predict *what is thrown next*.
> 2. **Predictive sequencing value** — prior pitches help predict the *outcome* of the current pitch,
>    after conditioning on the current pitch and game state.
> 3. **Prescriptive/causal value** — *changing* the sequence would improve outcomes.

WS4 tests whether *acting* on `q̂` beats the observed policy **within support** — the honest bridge
toward finding #3. It does **not** assume `q̂` is causal (`q̂(s,a) = E_model[R | s, A=a] ≠
E[R | s, do(A=a)]`); it reports `INCONCLUSIVE` when its estimators disagree. That firewall is WS3's,
restated here.

## The DATA_MODE toggle

This notebook is a **scaffold**. Phase 2 runs it on the real Statcast decision table; here a single
toggle, `DATA_MODE`, selects the world:

- `'synth_positive'` — the oracle's **positive world** (a planted ordered *outcome* effect: a
  velocity-transition whiff boost). **Default**, because it exercises the full myopic-ceiling story.
- `'synth_null'` — the oracle's **null world** (an ordered *selection* habit, no ordered *outcome*
  effect). Shows the count-driven-gain trap and the `O`-view overfitting signature live.
- `'real'` — the real decision table + WS3 artifacts built in Phase 2 (RUNBOOK Steps WS3, WS4.1).

## How to read this notebook

Every code step is bracketed by plain-worded markdown: **before** each cell we say what will happen
and why; **after** each cell we say how to read what came out. Numbers that depend on the real data
are `{PLACEHOLDER}` in the companion `PAPER.md`; here they simply appear when you run the cell. The
**Results** section (§9) is *branched* on four axes — **gate** (G-PASS/G-FAIL), **value**
(V+/V0/V−), **sequencing** (P+/P0/P−), and **verdict** (CONSISTENT/INCONCLUSIVE) — a code cell inspects
the computed report and prints which branch fired; the markdown that follows holds the pre-written
interpretation for every branch. The exact formulas live in `THEORY.md`; the exact code in
`workstreams/ws4_bandit/model.py` and `run_ws4.py`; the plain-English tour in `SEAN-README.md`.

> **Scale note.** WS4 *loads* WS3's models on real data and does no training, so it is much lighter
> than WS3. On a **synthetic** world, though, it **trains small WS3 stacks itself** (the C/L1/O
> behavior + outcome models it needs), so the cost here is that internal training — keep `N_GAMES`
> small for a quick pass. Feasibility (the SPEC §4 `≥50`-thrown mask) also needs enough games for
> synthetic pitchers to establish a repertoire, so a tiny `N_GAMES` yields empty masks (every row
> "carries no recommendation"); the default below is chosen to populate feasibility while staying
> quick. The **committed-validation** numbers quoted in the markdown are the ones to cite; a small
> in-notebook run reproduces the *verdicts*, not the exact CI widths.

## 1. Setup

In [ ]:
import sys
import platform
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- locate the repository root (works from repo root or from notebooks/) ---
REPO_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "pyproject.toml").exists() and (_p / "workstreams").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- shared foundation (WS0) ---
from pitchseq.config import load_config
from pitchseq.splits import make_splits
from pitchseq.families import FAMILIES
from pitchseq.eval import ope

# --- WS3 artifacts (the D33 supply WS4 consumes) ---
from workstreams.ws3_gbdt_stack.model import load_ws3_artifacts

# --- WS4 (this workstream) ---
from workstreams.ws4_bandit.model import (
    ABLATION_VIEWS, COMMON_EVAL_VIEW, POSTERIOR_SCALE, DEFAULT_N_SAMPLES, MU_FLOOR,
    thompson_policy, soften, ambiguity_stats, deviation_map, build_bandit_inputs, feasible_matrix,
)
# The runnable pipeline + two helpers reused for the in-notebook exhibits (no re-implementation):
from workstreams.ws4_bandit.run_ws4 import run_ws4, _format_headline, _feasibility_for_eval

CONFIG = load_config()
SEED = int(CONFIG.get("seeds", {}).get("global", 20260713))

# The world this run analyses: 'synth_positive' | 'synth_null' | 'real'.
# Default 'synth_positive': it plants a velocity-transition whiff boost, so the myopic-ceiling
# story (a state-value effect a bandit cannot cash in) is exercised end-to-end.
DATA_MODE = "synth_positive"

# WS4's prescriptive-ablation views (decision D38): C (context) -> L1 (prev pitch) -> O (full order).
# O is the richest view and the fixed COMMON evaluator; U / OM are out of scope for the myopic bandit.
VIEWS = list(ABLATION_VIEWS)                    # ("C", "L1", "O")
assert COMMON_EVAL_VIEW in VIEWS

# The pi_alpha grid (SPEC 9). Small + inclusive of 0 and 1 for a quick scaffold pass; the CLI uses
# the full config grid (ope.conservative_alpha). Value is linear in alpha, so 5 points draw the line.
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]

# Synthetic-world size + demo bootstrap/MC budgets (ignored when DATA_MODE == 'real'). WS4 trains
# small WS3 stacks internally on synth, so N_GAMES is the main cost; feasibility needs enough games.
N_GAMES = 200
WORLD_SEED = 7                 # the seed run_ws4 uses to build a synthetic world (its default)
THOMPSON_SEED = 20260714       # deterministic Thompson draws (run_ws4 default)
NB_N_BOOT = 100                # per-policy report bootstraps (CI width only)
NB_GAP_BOOT = 150              # clustered gap bootstraps (CI width only)
NB_N_SAMPLES = 600             # Thompson MC draws per row (MC error ~ 1/sqrt(N))

# Phase-2 real inputs (Step WS3 must have produced the artifacts; Step 1 the decision table).
WS3_DIR = REPO_ROOT / "results" / "ws3"
REAL_TABLE_PATH = REPO_ROOT / "data" / "processed" / "decision_table.parquet"

print(f"repo root : {REPO_ROOT}")
print(f"DATA_MODE : {DATA_MODE}")
print(f"views     : {VIEWS}   (common evaluator = {COMMON_EVAL_VIEW}; D38 prescriptive ablation)")
print(f"alphas    : {ALPHAS}")
print(f"N_GAMES   : {N_GAMES}   posterior_scale (kappa) = {POSTERIOR_SCALE}   n_samples = {NB_N_SAMPLES}")
print(f"seed      : {SEED}   world_seed : {WORLD_SEED}")

### Plotting style (fixed, colorblind-safe view colours)

In [ ]:
# Okabe-Ito qualitative palette (colorblind-safe) -- identical to WS1/WS2/WS3.
OKABE_ITO = {
    "orange":         "#E69F00",
    "sky_blue":       "#56B4E9",
    "bluish_green":   "#009E73",
    "yellow":         "#F0E442",
    "blue":           "#0072B2",
    "vermillion":     "#D55E00",
    "reddish_purple": "#CC79A7",
    "black":          "#000000",
}
# Fixed view -> colour, IDENTICAL to WS3 for C / L1 / O (WS4's ablation views). C is the
# context-only base; O is the fully ordered headline view; L1 the previous-pitch rung.
VIEW_COLORS = {
    "C":  OKABE_ITO["blue"],
    "L1": OKABE_ITO["bluish_green"],
    "O":  OKABE_ITO["vermillion"],
}
REF_COLOR = OKABE_ITO["black"]  # neutral: the behavior baseline V(mu) and reference lines

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "figure.autolayout": True,
})


def style_axes(ax):
    """Left+bottom spines only; no top/right. Returns the axis for chaining."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax


def new_fig(figsize=(7.2, 4.2)):
    """One figure, one axis, pre-styled."""
    fig, ax = plt.subplots(figsize=figsize)
    style_axes(ax)
    return fig, ax

## 2. What WS4 consumes — WS3's artifacts and the feasibility mask (D33)

WS4 reads **two things** and refits nothing (decision D33). From the shared decision table it takes
the reward `R = −delta_run_exp`, the leakage-safe feasibility mask, and the pitcher-game clustering.
From **WS3's saved artifacts** it takes, per state view, the behavior propensities `μ(a | s)`
(`BehaviorModel`), the counterfactual value grid `q̂(s, a)` for all 8 families (`OutcomeStack`), and
the residual `exp_reward_sd`. `build_bandit_inputs` assembles these into the aligned
`(q, q_sd, μ, feasible_mask, align)` bundle the pipeline evaluates.

**The fixed-yardstick design.** Every view's target policy is scored against **one common OPE
evaluator** — the behavior `μ` and `q̂` of the richest view `O` (`COMMON_EVAL_VIEW`). This is
deliberate: the `C → O` ablation asks whether the *information* in the ordered state improves the
*policy*; if each view were scored against its own `q̂`, a gap could reflect the evaluator changing
rather than the policy changing. Fixing the evaluator makes the gap a pure **information** statistic
(`THEORY.md` §5).

In [ ]:
def load_world(mode):
    """Return (decision_table, truth_meta) for the chosen world, matching run_ws4's world build."""
    if mode == "real":
        if not REAL_TABLE_PATH.exists():
            raise FileNotFoundError(
                f"real decision table not found at {REAL_TABLE_PATH}. Build it with "
                "`python -m pitchseq.build_table` (RUNBOOK Step 1) and run Step WS3 first."
            )
        return pd.read_parquet(REAL_TABLE_PATH, engine="pyarrow"), {"world": "real"}
    from pitchseq.decision_table import build_decision_table
    from pitchseq.synth import make_null_world, make_positive_world
    if mode == "synth_null":
        raw, truth = make_null_world(n_games=N_GAMES, seed=WORLD_SEED, innings_per_game=6)
    elif mode == "synth_positive":
        raw, truth = make_positive_world(n_games=N_GAMES, seed=WORLD_SEED, innings_per_game=6,
                                         effect_size=0.30, velo_gap_threshold=5.0)
    else:
        raise ValueError(f"unknown DATA_MODE {mode!r}")
    return build_decision_table(raw), truth


table, truth = load_world(DATA_MODE)
splits = make_splits(table, CONFIG)["primary"]
val = table.loc[splits["val"].to_numpy()].reset_index(drop=True)
test = table.loc[splits["test"].to_numpy()].reset_index(drop=True)
eval_rows = pd.concat([val, test], ignore_index=True) if len(test) else val.copy()

print(f"world      : {truth.get('world', DATA_MODE)}")
print(f"total rows : {len(table):,}   columns: {len(table.columns)}")
print(f"eval rows  : {len(eval_rows):,}   (validation + locked test — the rows WS3 scored)")
if DATA_MODE == "synth_positive":
    print(f"planted    : {truth.get('mechanism')} effect={truth.get('effect')} "
          f"threshold={truth.get('threshold')} empirical_whiff_lift={truth.get('empirical_whiff_lift')}")

### Run the full OPE-gated pipeline once

The heavy cell. It runs `run_ws4` end-to-end — the behavior-recovery gate, the Thompson targets per
view, the per-`α` OPE, and the `C → O` / `L1 → C` gaps — and returns the full report `R4`, which
every section below reads. On a synthetic world it **trains small WS3 stacks internally** and
checkpoints them (so we can reload the *same* stacks for the concrete-row illustration); on real data
it loads them from `--ws3-dir`. The FQE cross-check is off here for speed (it is verdict-excluded by
D24 anyway); the CLI runs it.

In [ ]:
def run_full_pipeline(mode):
    """Run run_ws4 for the chosen world; return (report, ws3_artifacts_dir)."""
    tmp = tempfile.mkdtemp(prefix="ws4_nb_")
    if mode == "real":
        if not WS3_DIR.exists():
            raise FileNotFoundError(f"WS3 artifacts not found at {WS3_DIR}; run Step WS3 first (D33).")
        R = run_ws4(source=str(REAL_TABLE_PATH), ws3_dir=str(WS3_DIR), synth="off", out=tmp,
                    views=VIEWS, alphas=ALPHAS, n_boot=NB_N_BOOT, gap_boot=NB_GAP_BOOT,
                    n_samples=NB_N_SAMPLES, seed=WORLD_SEED, fqe=False, write_outputs=True)
        return R, str(WS3_DIR)
    world = "positive" if mode == "synth_positive" else "null"
    R = run_ws4(synth=world, out=tmp, n_games=N_GAMES, views=VIEWS, alphas=ALPHAS,
                n_boot=NB_N_BOOT, gap_boot=NB_GAP_BOOT, n_samples=NB_N_SAMPLES,
                seed=WORLD_SEED, fqe=False, write_outputs=True)
    return R, tmp


R4, ART_DIR = run_full_pipeline(DATA_MODE)
AL = R4["alphas"]                       # the canonical alpha grid (float keys into R4["frontier"])
print(_format_headline(R4))

### The inputs WS3 supplies, per view — and the honest `κ` (posterior-scale) note

We reload WS3's `O`-view stack (the ones the pipeline just used) and build the bandit inputs to see
what WS4 actually consumes. Two honest points are made numerically here.

**The `κ` (POSTERIOR_SCALE) honesty note.** WS3's `exp_reward_sd` is a **residual** reward standard
deviation — the spread of a *single pitch's* reward around `q̂`, the **noise of baseball** (`~0.17`).
It is *not* the posterior standard error of the *mean* `q̂` that a Thompson sampler needs. Used raw it
is `~30×` the inter-family `q̂` gaps (`~0.005`) and drives the policy to a near-uniform draw.
`build_bandit_inputs` therefore returns `q_sd = κ · exp_reward_sd` with `κ = POSTERIOR_SCALE = 0.05`,
mapping the per-pitch residual sd to a posterior SE of a mean via `κ ≈ 1/√(n_eff)` (a LightGBM leaf
aggregates `n_eff ≈ 400` comparable pitches). `κ = 1` recovers the literal "`q̂ ± sd`" of decision
D36 but yields the uniform policy; `κ = 0.05` makes the Thompson uncertainty commensurate with the
`q̂` gaps. It is the single confidence knob, and we report it rather than hide it in the sampler.

**The feasibility-mask fix.** The SPEC §4 mask must be computed on the **full** table's trailing
window and subset to the evaluation rows — a 2024 game's trailing 365 days lie in the train seasons.
Computing it on the held-out rows *alone* undercounts the pre-game repertoire (it spuriously flagged
27% of decisions as no-recommendation before the fix). We use `_feasibility_for_eval` (the pipeline's
own full-table mask) here.

In [ ]:
artifacts = load_ws3_artifacts(ART_DIR)
feas_eval, low_hist_eval = _feasibility_for_eval(table, eval_rows, CONFIG)   # SPEC 4 full-table mask
inO = build_bandit_inputs(eval_rows, artifacts, COMMON_EVAL_VIEW, config=CONFIG,
                          posterior_scale=POSTERIOR_SCALE,
                          feasible_mask=feas_eval, low_history=low_hist_eval)

feas = inO.feasible_mask
n_feas = feas.sum(axis=1)
scaled_sd = float(np.nanmean(inO.q_sd[feas])) if feas.any() else float(np.nanmean(inO.q_sd))
residual_sd = scaled_sd / POSTERIOR_SCALE              # q_sd = kappa * exp_reward_sd (exact)
q_gap = R4["ambiguity"][COMMON_EVAL_VIEW].get("mean_gap", float("nan"))

print(f"q grid shape        : {inO.q.shape}   (n_eval x 8 families, FAMILIES order)")
print(f"mu floored/renorm   : rows sum to 1 (floor MU_FLOOR={MU_FLOOR:g}); min mu = {inO.mu.min():.2e}")
print(f"feasible / row      : mean {n_feas.mean():.2f}   empty-mask (no-rec) {float((n_feas==0).mean()):.1%}   "
      f"low-history {float(inO.align['low_history'].mean()):.1%}")
print(f"kappa (POSTERIOR_SCALE) = {POSTERIOR_SCALE}")
print(f"  residual exp_reward_sd (noise of baseball) ~ {residual_sd:.3f}")
print(f"  scaled  q_sd (posterior SE of the mean)    ~ {scaled_sd:.4f}   (= kappa * residual)")
print(f"  inter-family q_hat gap (top - runner-up)   ~ {q_gap:.4f}   -> gap/q_sd ~ {q_gap/scaled_sd:.2f}")
print(f"  (raw kappa=1 would give gap/sd ~ {q_gap/residual_sd:.3f}: normals overlap -> near-uniform policy)")

*Caption.* The `q̂` grid is an `(n × 8)` table — the model's expected run value for each family,
if it were the one thrown — a *conditional expectation*, **not** a causal effect (the firewall). The
`κ` block is the honest core: the raw residual sd (`~0.17`) is the noise of an individual pitch, so
the gap-to-sd ratio is tiny (`~0.03`) and Thompson sampling would shrug uniformly; the `κ = 0.05`
shrink turns it into the SE of the *mean* (`~0.008`), a gap-to-sd ratio near `~0.6` that yields a
*concentrated* recommendation. The empty-mask fraction is the share of low-history decisions that
carry **no recommendation** (the target mirrors the observed action there); with the full-table mask
it is small at scale.

## 3. The Thompson target policy — the method

**The exact Monte-Carlo procedure (decision D36).** For each decision row, `thompson_policy` draws
`n_samples` independent samples of the per-action value from `𝒩(q̂(s, a), q_sd(s, a)²)`, restricts the
`argmax` to the row's **feasible** actions (infeasible actions are pushed to `−∞` so they never win),
and returns the empirical win frequency per action:

$$\tilde\pi(a \mid s) = P\!\left(a = \arg\max_{a' \in \mathcal{F}(s)} \theta_{a'}\right),
\qquad \theta_{a'} \sim \mathcal{N}\!\big(\hat q(s, a'),\, \sigma(s, a')^2\big).$$

The result sums to exactly 1, places **zero** mass on infeasible actions, and is deterministic under a
seed. The Monte-Carlo error of each probability is `O(1/√n_samples)` — negligible against the OPE CI
widths that drive the verdict.

**Fallback rules.** A dominant mean yields a near-point-mass; equal means with equal sds yield a
uniform draw over the feasible actions; a row with an **empty** feasible mask (a low-history pitcher)
**falls back to the observed-action point mass** — it *carries no recommendation*, the target simply
mirrors what was thrown.

**Softening (`π_α`, SPEC §9).** The target is mixed toward behavior, `π_α = (1−α)μ + απ̃`, over the
`α` grid; `soften` delegates to `eval/ope.pi_alpha` (WS4 does not re-implement the mixture). The value
is linear in `α`, `V(π_α) = (1−α)V(μ) + αV(π̃)` (`THEORY.md` §4), so the frontier is a straight line.

**The D38 prescriptive-ablation design.** Building `π̃` from the `C`, `L1`, and `O` views' `q̂` — and
scoring all three under the *one fixed evaluator* — turns the state-view ladder into a prescription:
`V(π̃` from view-`v` `q̂`) across `v ∈ {C, L1, O}` is **the bandit's own falsification**. The `C → O`
gap is the sequencing-prescription evidence, isolated from the count-driven value-vs-behavior gain.

In [ ]:
# One concrete decision row (sequence-eligible + decidable, so the ordered view is engaged).
elig = (eval_rows["pitch_number"].to_numpy() >= 2)
cand = np.flatnonzero((n_feas >= 2) & elig)
if len(cand) == 0:
    cand = np.flatnonzero(n_feas >= 2)

if len(cand) == 0:
    print("no decidable rows at this N_GAMES (feasibility empty) -- raise N_GAMES to see a recommendation.")
else:
    ridx = int(cand[0])
    q_row, sd_row, f_row, mu_row = inO.q[ridx], inO.q_sd[ridx], inO.feasible_mask[ridx], inO.mu[ridx]
    pi_row = thompson_policy(q_row[None, :], sd_row[None, :], f_row[None, :],
                             n_samples=NB_N_SAMPLES, rng=THOMPSON_SEED)[0]

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(10.5, 4.0))
    x = np.arange(len(FAMILIES))
    colors = [OKABE_ITO["vermillion"] if f_row[i] else "#cccccc" for i in range(len(FAMILIES))]
    axL.bar(x, q_row, color=colors); style_axes(axL)
    axL.set_xticks(x); axL.set_xticklabels(FAMILIES); axL.set_title("q̂(s, a): the what-if card (feasible = solid)")
    axL.set_ylabel("E[R | s, a]  (runs)"); axL.axhline(0, color=REF_COLOR, lw=0.8)
    axR.bar(x, pi_row, color=colors); style_axes(axR)
    axR.set_xticks(x); axR.set_xticklabels(FAMILIES); axR.set_title("π̃(a | s): Thompson P(argmax), feasible only")
    axR.set_ylabel("recommendation probability")
    plt.show()
    top = int(np.argmax(pi_row))
    print(f"row {ridx}: {int(f_row.sum())} feasible families; top recommendation = {FAMILIES[top]} "
          f"at pi_tilde = {pi_row[top]:.2f}")

*Caption.* Left: the `q̂` what-if card — the model's expected run value for each family, feasible
ones solid. Right: the Thompson recommendation `π̃`, the share of imagined worlds each feasible family
wins. A top probability far from 1 is the honest signal that "best next pitch" is a *toss-up* here —
several families are within their error bars of each other (the ambiguity exhibit, §8, quantifies how
often).

In [ ]:
# pi_alpha: the conservative dial from behavior (alpha=0) to the pure target (alpha=1), one row.
if len(cand) > 0:
    fig, ax = new_fig((7.2, 4.0))
    feas_idx = np.flatnonzero(f_row)
    for i in feas_idx:
        ys = [soften(pi_row[None, :], mu_row[None, :], a)[0][i] for a in ALPHAS]
        ax.plot(ALPHAS, ys, marker="o", label=FAMILIES[i])
    ax.set_xlabel("alpha  (0 = behavior mu,  1 = Thompson target)")
    ax.set_ylabel("policy probability  pi_alpha(a | s)")
    ax.set_title("pi_alpha interpolates each family from mu to the target")
    ax.legend(fontsize=8, ncol=2, frameon=False)
    plt.show()
    print("At every alpha the row is a valid distribution (a convex mix of two distributions); "
          "the value moves linearly, the ESS does not.")

*Caption.* Each feasible family's probability moves in a straight line from its behavior value at
`α = 0` to its Thompson value at `α = 1` — the conservative mixture. The *value* of the policy moves
linearly too (`THEORY.md` §4); what does **not** is the effective sample size, which degrades as `α`
rises (the price of deviating from behavior, read in §5).

## 4. The OPE gate — behavior-policy recovery, FIRST

Before any target value, the pipeline re-runs the SPEC §9 behavior-policy recovery self-test: set the
target equal to the behavior policy and require every estimator to recover the observed held-out mean
reward, with the IPS importance weights exactly 1. This is the gate — it runs *first*, and everything
below it depends on it passing.

In [ ]:
rec = R4["behavior_recovery"]
gate = R4["gate"]
print("=" * 64)
print(f" OPE GATE  (SPEC 0.3 / D37):  {gate}")
print("=" * 64)
print(f" observed held-out mean reward : {rec.get('observed_mean'):+.4f}")
print(f" IPS weights exactly unit      : {rec.get('ips_weights_unit')}   (pi_0 == mu => w == 1)")
for name, e in rec.get("estimators", {}).items():
    within = e.get("within_ci")
    val = e.get("value")
    print(f"   {name:<10} value={val:+.4f}" + (f"  within_ci={within}" if within is not None else ""))
print("-" * 64)
print(f" passed = {rec.get('passed')}  ->  " +
      ("gate PASSED; target values below are interpretable." if gate == "PASSED"
       else "FAILED_GATE; STOP -- nothing below is trustworthy."))

**The rule (verbatim discipline).** If this prints **`FAILED_GATE`**, the OPE harness cannot
recover even the *observed* policy's value on this data. **Stop.** No target value below the gate is
trustworthy — paste the `FAILED_GATE` block and diagnose the propensity / reward join before reading
anything else. `IPS weights unit = True` confirms `π_0 = μ` gave exactly unit importance weights, the
tell-tale that "recommend what he already does" was scored as identical to what he did. This is SPEC
§0.3 enforced *structurally*, not a soft warning.

## 5. Per-view, per-`α` evaluation — the value-vs-`α` frontier

The SPEC §9 report block for every `(view, α)`: the policy value, its one-sided 95% lower bound, the
effective sample size, the out-of-support fraction, the max importance weight, and the per-`α`
agreement verdict (DM/SNIPS/DR, decision D24). The frontier reads the value against a **degrading
support overlay** — a stronger recommendation (`α → 1`) costs effective sample size.

In [ ]:
hdr = f"   {'view':<4} {'alpha':>5} {'value':>9} {'lower95':>9} {'d(vs beh)':>9} {'ESS%':>6} {'oos':>6} {'maxw':>6} {'verdict':>12}"
Vmu = R4["behavior_value"]
print(f" behavior value V(mu) = {Vmu:+.4f}   (alpha = 0 baseline)")
print(hdr)
for v in VIEWS:
    for a in AL:
        c = R4["frontier"][v][a]
        dv = c["value"] - Vmu if c["value"] is not None else float("nan")
        print(f"   {v:<4} {a:>5.2f} {c['value']:>+9.4f} {c['lower_95']:>+9.4f} {dv:>+9.4f} "
              f"{100*c['ess_frac']:>5.1f}% {c['oos_support_frac']:>6.2f} {c['max_weight']:>6.1f} {str(c['verdict']):>12}")
print("   (D39: INCONCLUSIVE and lower95 < V(mu) are first-class honest results, not failures.)")

In [ ]:
# The value-vs-alpha frontier: value + one-sided lower_95 band per view, with an ESS% overlay.
fig, ax = new_fig((8.2, 4.6))
ax2 = ax.twinx()
for v in VIEWS:
    ys = np.array([R4["frontier"][v][a]["value"] for a in AL], float)
    los = np.array([R4["frontier"][v][a]["lower_95"] for a in AL], float)
    ess = np.array([100 * R4["frontier"][v][a]["ess_frac"] for a in AL], float)
    ax.plot(AL, ys, color=VIEW_COLORS[v], marker="o", label=f"{v} value")
    ax.fill_between(AL, los, ys, color=VIEW_COLORS[v], alpha=0.12)
    ax2.plot(AL, ess, color=VIEW_COLORS[v], ls="--", lw=1.2, alpha=0.7)
ax.axhline(Vmu, color=REF_COLOR, ls=":", lw=1.2, label="V(mu) behavior")
ax.set_xlabel("alpha  (conservative mixing toward the Thompson target)")
ax.set_ylabel("OPE value  (runs)  -- solid; shaded = one-sided 95% lower band")
ax2.set_ylabel("ESS %  (dashed)"); ax2.set_ylim(0, 105)
style_axes(ax); ax2.spines["top"].set_visible(False)
ax.legend(loc="best", fontsize=8, frameon=False)
ax.set_title("Value-vs-alpha frontier (fixed O evaluator) with ESS overlay")
plt.show()

**How to read ESS / support / weights.** `ESS%` must fall as `α` rises (100% at `α = 0`, lower at
`α = 1`) — the price of deviating from behavior; a collapse to a few percent at moderate `α` means the
target is far outside behavior support and its value should be read with suspicion. `oos` (mean target
mass on actions with `μ < 1%`) should stay small; a large value means the policy recommends
rarely-thrown actions the OPE cannot evaluate. On the committed validation the ESS falls `100% → ~45–51%`
as `α → 1` and the max weight rises to `~15` — wide but not collapsed. Because the value is linear in
`α`, the solid lines are straight; the *bands* widen with `α` as the ESS shrinks.

## 6. The prescriptive-ablation table — the heart

The `C → O` (and `L1 → C`) value gap is the **sequencing-prescription** statistic: a pitcher-game
clustered **paired** bootstrap of the doubly-robust per-row contributions under the *common* evaluator
(the pairing cancels the common-evaluator variance, `THEORY.md` §7). It is *not* the raw
value-vs-behavior column — that is contaminated by count-driven gains.

In [ ]:
print(" SEQUENCING-PRESCRIPTION GAPS (clustered by pitcher-game; common evaluator = O):")
print(f"   {'alpha':>5} {'O - C':>9} {'CI_lo':>9} {'CI_hi':>9} {'lower95':>9} {'L1 - C':>9}")
for a in AL:
    g = R4["gaps"][a]
    oc = g["O_minus_C"]
    l1 = g.get("L1_minus_C", {})
    flag = "  <-- CI excludes 0" if oc["lower_95"] > 0 else ""
    print(f"   {a:>5.2f} {oc['value']:>+9.4f} {oc['ci95'][0]:>+9.4f} {oc['ci95'][1]:>+9.4f} "
          f"{oc['lower_95']:>+9.4f} {l1.get('value', float('nan')):>+9.4f}{flag}")

**Completed validation (state these as done — the committed WS4a run).** Both worlds pass the
gate (behavior recovery exact, IPS weights ≡ 1 at `α = 0`).

- **Null world** — `V(μ) = +0.0047`. The `O − C` gap is **significantly negative**: `−0.0015`
  (CI `[−0.0028, −0.0001]`) at `α = 0.5` and `−0.0030` (CI `[−0.0056, −0.0002]`) at `α = 1`. The
  `O`-view `q̂` **overfits** — extra ordered features add estimation variance in a world with no ordered
  outcome effect, so the `O`-built policy is slightly worse under the fixed evaluator (the prescriptive
  echo of WS3's negative `Δ_matchup`). `L1 − C ≈ +0.0002`. Verdict: **`SEQ_NEUTRAL_PRESCRIPTION`**.
- **Positive world** — `V(μ) = +0.0111`. The `O − C` gap is `≈ +0.0001` with CIs that **straddle zero**
  at `α = 0.5/1` — but sits `~+0.003` **above** the null world's `O − C`. That offset is the sequencing
  signal exactly cancelling the `O`-view overfitting, leaving a gap near zero. Verdict:
  **`SEQ_INCONCLUSIVE_MYOPIC`** (§7).

**The count-driven-gain trap.** On both worlds the bandit beats the *habit-based* behavior policy in
raw value (the `d(vs beh)` column in §5). That is **not** sequencing evidence: the synthetic behavior
policy is not reward-optimal, so a myopic count-aware policy improves on it for reasons that have
nothing to do with order (value is dominated by the count; Tango et al. 2007). Only the `C → O` gap
isolates sequencing — which is why the paper leads with the gap, not the raw column.

In [ ]:
presc = R4.get("prescription")
if presc:
    print(f" D38 synthetic verdict : {presc['verdict']}")
    oc = presc["o_minus_c_upper"]
    print(f"   O - C at alpha={presc['upper_bound_alpha']:.2f} (upper bound): "
          f"{oc['value']:+.4f}  lower95={oc['lower_95']:+.4f}")
    if presc.get("count_driven_gain"):
        print(f"   NOTE: the bandit beats behavior by up to {presc['value_over_behavior_max']:+.4f} in value,")
        print("   but the synthetic behavior policy is habit-based (not reward-optimal): this COUNT-DRIVEN")
        print("   improvement is real yet is NOT sequencing evidence. Only the O - C gap isolates it (D38).")
else:
    print("real-data run: no synthetic ground-truth verdict; read the C -> O gap CI above (branch P in §9).")

## 7. The myopic ceiling — the intellectual payload

Why is the positive world's honest verdict `SEQ_INCONCLUSIVE_MYOPIC` and not a failure? Because the
planted effect is a **state-value** effect, and a myopic bandit can reach only a measured sliver of it.

**The decomposition (`THEORY.md` §6).** Split the per-pitch expected reward into a **state** term and
an **action-differential** term,

$$\mathbb{E}[R \mid s, a] = v(s) + \delta(s, a),$$

where `v(s)` depends only on the state and `δ(s, a)` is the part that differs across *today's* action.
A myopic `argmax` policy's edge over an action-blind policy depends **only on `δ`** — `v(s)` is added to
every action's value and cancels in the argmax.

**Where the planted effect lives.** The trigger `|velo_{t−1} − velo_{t−2}| ≥ 5` is fixed by the
**history**, so by the current decision the `+0.032` whiff boost is already owed to `v(s)` — **every**
feasible current family inherits it equally (ground-truth probes confirm the uniform `+0.032` lift). A
greedy chooser cannot manufacture it. It can harvest only the **family-differential**: split-finger
`δ ≈ +0.042` vs four-seam `δ ≈ +0.030`, a spread of `~0.012`, worth a best-case `~0.003`-run `O − C`
edge on the triggered rows. And `~0.003` sits **below the OPE noise floor** (`~0.006` CI half-width at
this scale), so the gap is undetectable *by construction*.

**The setup term (visible only to sequential planners).** The other order of magnitude is the
**setup**: throwing the pitch that *creates* the velocity transition a pitch early, which pays off in
the *next* state — the term `γ·E[v(s′) | s, a]` a bandit's one-step value (`γ = 0`) **drops**. WS5's
MDP and WS7's offline RL keep it and can value the setup; a bandit provably cannot. **Decision D40**:
their positive-world acceptance is to **exceed the `~0.003` ceiling**.

In [ ]:
# The myopic-ceiling decomposition (committed ground-truth probes; decision D40). These are
# constants from the generative model + probes, NOT read from this small demo run.
STATE_LIFT   = 0.032   # uniform whiff-boost owed to v(s) on a triggered row -- every family gets it
DELTA_FS     = 0.042   # split-finger family-differential on a triggered row
DELTA_FF     = 0.030   # four-seam family-differential
DELTA_SPREAD = DELTA_FS - DELTA_FF          # ~0.012 -- the myopically-visible spread
MYOPIC_REACH = 0.003   # best-case O - C myopic policy edge (differential harvested on triggered rows)
NOISE_FLOOR  = 0.006   # OPE gap CI half-width at synthetic scale

fig, ax = new_fig((8.6, 4.2))
labels = ["planted state lift\nv(s)  (uniform)", "family spread\nδ (FS − FF)",
          "myopic reach\nO − C edge", "OPE noise floor\n(CI half-width)"]
vals = [STATE_LIFT, DELTA_SPREAD, MYOPIC_REACH, NOISE_FLOOR]
cols = [OKABE_ITO["reddish_purple"], OKABE_ITO["orange"], OKABE_ITO["vermillion"], OKABE_ITO["black"]]
bars = ax.bar(np.arange(len(vals)), vals, color=cols)
ax.set_xticks(np.arange(len(vals))); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("runs")
ax.set_title("The myopic ceiling: greed reaches δ (~0.003), not v(s) (~0.032)")
for b, val in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, val + 0.0008, f"{val:.3f}", ha="center", fontsize=9)
ax.axhline(NOISE_FLOOR, color=REF_COLOR, ls=":", lw=1.0)
plt.show()
print(f"myopically-reachable edge ~{MYOPIC_REACH:.3f} < OPE noise floor ~{NOISE_FLOOR:.3f}  "
      f"=> gap CI contains 0 by construction (undetectable at this scale).")
print(f"the reachable {MYOPIC_REACH:.3f} is {MYOPIC_REACH/STATE_LIFT:.0%} of the planted {STATE_LIFT:.3f}; "
      "the rest is the SETUP a sequential planner (WS5/WS7) must value.")

**The gap-machinery self-test (the machinery is not blind).** To prove the near-zero gap is the
*effect's* property and not a broken detector, WS4a ships a self-test that constructs a world with a
genuine *myopic* family advantage and confirms the `SEQ_EXPLOITED` path fires — the `O − C` lower bound
clears 0 when there *is* a myopic edge to find. So the machinery **sees**; the positive world's effect
is genuinely **sequential-not-myopic**.

**The positive-vs-null discrimination.** Discrimination is visible even though the absolute claim is
not clearable: the positive world's `O − C` sits `~+0.003` **above** the null world's *significantly
negative* gap. The `~0.003` of real prescriptive work is present (WS3 recovered the effect at ratio
0.955) — it just lands below the myopic OPE floor, exactly cancelling the `O`-view overfitting the null
world exposes.

**The ladder consequence.** A myopic policy cannot value a **setup** pitch. That is not a bug in WS4;
it is a structural property of the *class* of policy, and WS4 measures exactly how far it gets (`~0.003`
of `~0.032`). It is precisely what **WS5** (a tabular MDP that assigns credit to a setup pitch through
the count transition) and **WS7** (offline RL with the full OPE battery) exist to test — their target
(D40) is to **exceed this ceiling**.

## 8. Ambiguity & deviation exhibits

Two honest exhibits about *how resolvable* myopic pitch prescription actually is, and *where* the
target dares to differ from behavior.

In [ ]:
# Ambiguity: for each decidable row, the posterior probability the top family beats the runner-up.
amb_live = ambiguity_stats(inO.q, inO.q_sd, inO.feasible_mask)      # live call on the O inputs
amb_R4 = R4["ambiguity"][COMMON_EVAL_VIEW]                           # pipeline's own (should match)
print(f" O-view ambiguity  mean P(top > runner-up) = {amb_R4['mean_p_beat']:.3f}   "
      f"(decidable={amb_R4['n_decidable']}, no-rec={amb_R4['n_infeasible']})")
print(f"   share of recommendations that are TOSS-UPS at 50 / 80 / 95% confidence = "
      f"{amb_R4['ambiguous_50']:.2f} / {amb_R4['ambiguous_80']:.2f} / {amb_R4['ambiguous_95']:.2f}")
print(f"   (live cross-check on rebuilt inputs: mean_p_beat={amb_live['mean_p_beat']:.3f})")

fig, ax = new_fig((7.6, 4.0))
levels = ["ambiguous_50", "ambiguous_80", "ambiguous_95"]
xlab = ["@50%", "@80%", "@95%"]
xs = np.arange(len(levels)); w = 0.25
for k, v in enumerate(VIEWS):
    s = R4["ambiguity"][v]
    ys = [s[L] if s[L] == s[L] else 0.0 for L in levels]      # NaN-safe (empty-mask worlds)
    ax.bar(xs + (k-1)*w, ys, width=w, color=VIEW_COLORS[v], label=v)
ax.set_xticks(xs); ax.set_xticklabels(xlab)
ax.set_ylabel("share of recommendations that are toss-ups")
ax.set_title("Ambiguity: how often the 'best next pitch' is not a confident distinction")
ax.legend(frameon=False)
plt.show()

**The `~99%`-toss-up honesty.** On the committed validation the mean posterior confidence that the
top family beats the runner-up is only `~0.62–0.66`, and `~99%` of recommendations are toss-ups at 95%
confidence. This is not a bug — it is an honest statement of how resolvable "the single best next pitch"
is under myopia at family resolution: several families are within their error bars of each other. The
correct practitioner takeaway is a **shortlist, not a pick** — "these families are near-equivalent
here," which is exactly what the Thompson distribution already gives. A confident single pick would be
over-reading the `q̂` gaps; the uncertainty machinery exists to prevent exactly that.

In [ ]:
# Deviation-from-behavior heatmaps by count (mean total-variation distance at alpha = 1).
def dev_grid(view):
    grid = np.full((4, 3), np.nan)      # balls 0..3 x strikes 0..2
    for r in R4["deviation"][view]["by_count"]:
        b, s = int(r["balls"]), int(r["strikes"])
        if 0 <= b < 4 and 0 <= s < 3:
            grid[b, s] = r["mean_tv"]
    return grid

fig, axes = plt.subplots(1, len(VIEWS), figsize=(11.5, 3.4))
vmax = max((R4["deviation"][v]["overall_mean_tv"] for v in VIEWS), default=0.5) * 1.6 or 0.5
for ax, v in zip(np.atleast_1d(axes), VIEWS):
    g = dev_grid(v)
    im = ax.imshow(g, origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=vmax)
    ax.set_title(f"{v}: mean TV = {R4['deviation'][v]['overall_mean_tv']:.3f}", fontsize=10)
    ax.set_xlabel("strikes"); ax.set_ylabel("balls")
    ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2, 3])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Where the target dares to differ from behavior (TV distance by count, alpha = 1)", y=1.04)
plt.show()

**What a practitioner should take from this.** The deviation maps show *where* the recommendation
departs from what the pitcher already does — brighter count cells are where the bandit most wants to
change the mix. On the committed validation the overall mean TV at `α = 1` is `~0.35` for `C` and `~0.30`
for `O` (the ordered policy stays slightly *closer* to behavior, its Thompson draws being less
concentrated). Read alongside the ambiguity exhibit: the bandit deviates most where the count most
constrains value — but because `~99%` of those deviations are toss-ups, the honest product is a *shift
in the distribution*, not a confident swap of one pitch for another.

## 9. Results — branched interpretation (G / V / P / verdict)

The result is read on **four axes**: the **gate** (a precondition), the **value** and **verdict** axes
(how much to trust the numbers), and the **sequencing** axis (WS4's actual question). The code cell
inspects `R4` and prints which branch fired on each; the pre-written markdown below holds the
interpretation for every branch.

In [ ]:
AL = R4["alphas"]
a_mod = 0.5 if 0.5 in AL else AL[len(AL) // 2]           # a moderate deviation level
a_top = AL[-1]
Vmu = R4["behavior_value"]

# --- Gate axis ---
gate_branch = "G-PASS" if R4["gate"] == "PASSED" else "G-FAIL"

# --- Value axis (at moderate alpha, common evaluator O, vs V(mu)) ---
c = R4["frontier"][COMMON_EVAL_VIEW][a_mod]
lo, hi = c["ci95"]
if c["lower_95"] > Vmu:
    value_branch = "V+"
elif hi < Vmu:
    value_branch = "V-"
else:
    value_branch = "V0"

# --- Sequencing axis (O - C gap at the upper-bound alpha) ---
g = R4["gaps"][a_top]["O_minus_C"]
if g["lower_95"] > 0:
    seq_branch = "P+"
elif g["ci95"][1] < 0:
    seq_branch = "P-"
else:
    seq_branch = "P0"

# --- Verdict axis (per-alpha DM/SNIPS/DR agreement, D24) ---
verds = [R4["frontier"][v][a]["verdict"] for v in VIEWS for a in AL]
n_incon = sum(x == "INCONCLUSIVE" for x in verds)
verdict_axis = "INCONCLUSIVE" if n_incon > 0 else "CONSISTENT"

print("=" * 68)
print(" WS4 RESULTS -- branch selector")
print("=" * 68)
print(f" world / mode        : {R4['world']}")
print(f" behavior value V(mu): {Vmu:+.4f}    gate: {R4['gate']}")
print(f" O value @ alpha={a_mod:.2f} : {c['value']:+.4f}  (lower95 {c['lower_95']:+.4f}, CI [{lo:+.4f},{hi:+.4f}])")
print(f" O - C gap @ alpha={a_top:.2f}: {g['value']:+.4f}  (CI [{g['ci95'][0]:+.4f},{g['ci95'][1]:+.4f}], lower95 {g['lower_95']:+.4f})")
print("-" * 68)
print(f" GATE   axis : {gate_branch}")
print(f" VALUE  axis : {value_branch}   (at moderate alpha={a_mod:.2f})")
print(f" SEQ    axis : {seq_branch}     (the real question -- O vs C)")
print(f" VERDICT axis: {verdict_axis}   ({n_incon}/{len(verds)} per-alpha cells INCONCLUSIVE)")
if R4.get("prescription"):
    print(f" synthetic verdict : {R4['prescription']['verdict']}")
print("=" * 68)
print(" -> read the matching branch write-ups in the markdown below.")

### Gate axis

**G-PASS — behavior recovery passes.** The OPE harness recovers the observed policy's value (IPS
weights ≡ 1 at `α = 0`); every target value below the gate is interpretable. Proceed.

**G-FAIL — `FAILED_GATE`.** The harness cannot recover even the *observed* policy's value on this data.
**Stop.** No target value is trustworthy; paste the `FAILED_GATE` block and diagnose the propensity /
reward join before reading anything else. SPEC §0.3, enforced structurally.

### Value axis (at moderate `α`, read against `V(μ)`)

**V+ — `lower_95 > V(μ)`: the bandit beats behavior.** The moderate-`α` policy's 95% lower bound clears
the behavior value — a real improvement over the observed policy *in myopic value*. Before believing it,
check the support (ESS not collapsed, oos small, max weight moderate); a "gain" riding on a few
high-weight rows is an artifact. This is **not** by itself sequencing evidence — the habit-based
behavior policy is beatable for count-driven reasons (read the P axis).

**V0 — CI straddles `V(μ)`: the typical honest outcome.** The policy value is indistinguishable from
behavior at this scale — the expected result for a conservative myopic recommender on real baseball,
where value is dominated by the count and the room to improve *myopically* is thin.

**V− — `lower_95 < V(μ)`: the policy trails behavior.** The target scores *below* the observed policy — a
`q̂` or propensity misfit, or a support problem (the target recommends actions the OPE cannot evaluate).
Diagnose with the oos fraction and max weight; do not report a prescription from a policy that loses to
behavior.

### Sequencing axis (`O − C` gap — the real question)

**P+ — `O − C` gap CI `> 0`: prescriptively exploitable ordering.** The ordered-state policy beats the
context-only policy under the fixed evaluator by more than the clustered CI — the ordered state carries
information a *myopic* recommender can *act on*. This would **exceed the myopic ceiling** the synthetic
world measures, so it is a strong claim: cross-check it against WS3's order axis (a P+ here should have an
`H1` there — predictable *and* exploitable order), confirm it concentrates in `long_pa` / `two_strike`
slices, and confirm the per-`α` verdict is `CONSISTENT`. If it survives, the burden passes to WS7's full
OPE battery.

**P0 — `O − C` gap `≈ 0` (CI contains 0): two sub-readings.** The ordered-state policy is
indistinguishable from the context-only policy. Which of two things it means is resolved by the
positive-vs-null discrimination check (§7) — does the real-data `O − C` sit meaningfully **above** the
null-world overfitting signature?
- *Nothing there.* If the gap is near the (negative) overfitting baseline, there is no sequencing
  prescription signal — order is either not predictive of outcomes (cross-check WS3's `H2/H3`) or
  predictive-but-not-exploitable.
- *Sequential-not-myopic.* If the gap sits **above** the overfitting baseline (the positive world's
  `~+0.003` offset) but its own CI still contains 0, the sequencing signal is *present but below the
  myopic OPE floor* — the WS4 fixture's own verdict (`SEQ_INCONCLUSIVE_MYOPIC`). Route it to WS5/WS7: the
  effect is a setup a bandit cannot cash in, and the sequential rungs' acceptance is to exceed exactly
  this ceiling (D40).

**P− — `O − C` gap CI `< 0`: `O`-view `q̂` overfitting dominates.** The ordered-state policy is
significantly *worse* than the context-only policy under the fixed evaluator — the null-world signature.
The extra ordered features in `O`'s `q̂` add estimation variance without prescriptive signal, so the
`O`-built policy fragments (the prescriptive echo of WS3's negative `Δ_matchup`). Read it as "no
sequencing prescription edge **and** a real cost of building from the ordered `q̂`," and build the policy
from the simpler view.

### Verdict axis (per-`α`, decision D24) — and reading the grid

**CONSISTENT.** DM, SNIPS, and DR agree within the wider of their CI half-widths at this `α`; the value
and its lower bound are a coherent read.

**INCONCLUSIVE.** The monitored estimators disagree materially. Per SPEC §9's closing rule the verdict is
**INCONCLUSIVE, not "it works"**: the value is not a coherent read at this `α` (typically a high-`α`,
low-ESS row where the model-based DM and the weighted DR pull apart). Lean on the lower-`α`, higher-ESS
rows.

**Reading the grid.** The honest headline is a tuple `(G, V, P, verdict)`. The *most anticipated* cell is
**G-PASS × V0 × P0(sequential-not-myopic) × CONSISTENT** at moderate `α` — the WS4 fixture's own reading,
and the one that motivates the sequential rungs. The *strongest* cell is **G-PASS × V+ × P+ ×
CONSISTENT** — a myopically-actionable ordered edge, cross-checked hard against WS3 and WS7. The
*diagnostic* cells are any with **G-FAIL** (stop) or **V−/P−** (misfit / overfitting — diagnose before
reading).

## 10. Discussion and limitations

**The ceiling as the bridge to WS5/WS7.** WS4's contribution is a *measurement of a limit*, not a run
gain. A myopic bandit can exploit only the `δ(s, a)` part of an effect — the part that differs across
today's action — which on the positive world is `~0.003` of a `~0.032` planted effect, below the OPE
noise floor, with the gap-machinery self-test proving the near-zero measurement is the effect's property.
The remaining order of magnitude is the **setup** — value that accrues in a future state — which a
sequential model can see and a bandit cannot. Decision D40 makes the handoff falsifiable: WS5 and WS7 pass
their positive-world acceptance by **exceeding 0.003**.

**Myopia as a structural limit (not a bug).** The ceiling is a property of the *class* of policy. WS4's
honest output is exactly how far greed gets — which is the informative thing, not a shortfall of this
implementation.

**POSTERIOR_SCALE (`κ`) sensitivity.** The `κ = 0.05` shrink that turns residual noise into a posterior SE
(§2) is a modelling choice, justified by the leaf-support argument but not learned. A smaller `κ` sharpens
the recommendations (fewer toss-ups); a larger `κ` softens them toward uniform. The ambiguity exhibit and
deviation maps are `κ`-sensitive; the *gap* and *verdict* are far less so, because the gap is a policy
*difference* under a fixed evaluator. We report `κ` explicitly rather than bury it in the sampler.

**The fixed-evaluator choice's tradeoffs.** Scoring every view against `O`'s evaluator makes the gap an
information statistic (§2), at the cost that the absolute value *level* of each view is expressed in the
`O`-evaluator's units — so the value-vs-behavior column is most trustworthy for `O` and read as a level,
not a per-view ranking, for `C`/`L1`. The gap, a difference under the same evaluator, is unaffected.

**Inheritance.** WS4 hands the ladder a measured ceiling and a clean question. **WS5** values setups
explicitly (a tabular MDP over the count transition); **WS7** does it with function approximation and the
full OPE battery (FQE/DR agreement, support diagnostics, the exploitability read-out). Both inherit WS4's
gate discipline and its honest-negative branches.

**Limitations.** (1) *Myopia* — structural, measured, the reason WS5/WS7 exist. (2) *`κ` choice* — a
justified but unlearned shrink; results reported with it explicit. (3) *Evaluator dependence* — value
levels are in the fixed `O`-evaluator's units (not the gap). (4) *Synthetic-scale OPE noise floor* — the
`~0.006` half-width that swallows the `~0.003` edge is a fixture-scale property; at full-data scale the
floor shrinks as `~1/√(ESS)` (`THEORY.md` §8), and whether the real-data `C → O` gap resolves above 0 is
the open question WS4 poses (a fixture-scale P0 does not preclude a full-scale P+). (5) *The firewall* —
WS4 tests whether acting on `q̂` beats behavior within support; it does not certify `q̂` as causal.
(6) *Family granularity and the `pitch_type` proxy* — inherited from WS3. (7) *One reward metric*
(`−delta_run_exp`).

## 11. Reproducibility appendix

**Phase-2 commands (RUNBOOK WS4.1).** WS4 **consumes WS3's artifacts** (decision D33) and trains nothing
on real data — **Step WS3 must have run first** (the per-view `behavior_<view>.joblib` and
`outcome_<view>.joblib` under `results/ws3/`).

```powershell
conda activate statcast; cd ~\pitch-sequencing-research
# WS4.1 -- Thompson target policy over WS3's q-hat, OPE-gated, per view C L1 O
python workstreams/ws4_bandit/run_ws4.py --table data/processed/decision_table.parquet --ws3-dir results/ws3/ --out results/ws4/ --views C L1 O
```

The synthetic Phase-1 CI equivalents (no data, no WS3 dependency -- WS4 fits small WS3 stacks itself and
checkpoints them under `--out`): `--synth null --out results/ws4_null/` and
`--synth positive --out results/ws4_pos/`. `--posterior-scale` (default 0.05) is the single Thompson
confidence knob; `--n-samples` (default 1500) the MC draw count; `--gap-boot` / `--n-boot` set the CI
widths; `--no-fqe` skips the (verdict-excluded, D24) count-conditional FQE cross-check.

**Artifact inventory (written under `results/ws4/`).**

- `policy_real_<view>.parquet` — standard-schema `policy_prob` predictions per view (the pure `α = 1`
  target).
- `frontier_real.csv` — the value-vs-`α` frontier overlay data (value, lower_95, ESS, support).
- `ws4_report_real.json` — the full report (gate, per-view/per-`α` OPE table, `C→O`/`L1→C` gaps with
  clustered CIs, deviation maps, ambiguity stats, verdict).
- `ws4_real.runmeta.json` — timing / peak RAM for the SPEC §7 Pareto plot.

**Dependency (explicit).** WS4 reads WS3's `WS3Artifacts` through `load_ws3_artifacts` and scores strictly
through `eval/ope.evaluate_policy`; it re-runs behavior-policy recovery in-pipeline before any target value
(SPEC §0.3 / D37).

In [ ]:
import scipy, sklearn, joblib
print("python     :", platform.python_version())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("scipy      :", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib     :", joblib.__version__)
print("matplotlib :", matplotlib.__version__)
print("seed       :", SEED, " world_seed:", WORLD_SEED, " thompson_seed:", THOMPSON_SEED)
print("data mode  :", DATA_MODE)
print("views      :", VIEWS, " common evaluator:", COMMON_EVAL_VIEW)
print("alphas     :", ALPHAS, " kappa:", POSTERIOR_SCALE, " n_samples:", NB_N_SAMPLES)